In [2]:
import pandas as pd
import duckdb
import matplotlib

df_raw = pd.read_excel('../data_files/brå_stockholm_21_25_raw.xls', header=None)

df_raw.head(10)


,0,1,2,3,4,5
0,Anmälda brott,NaN,Brottsförebyggande rådet,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,År,År,År,År,År
3,NaN,2021,2022,2023,2024,2025
4,NaN,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN,NaN,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",1004,978,1017,1039,1072
9,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN


In [4]:
# Döp om  kolumn '0' till 'Brottstyp'

df_cleaned = df_raw.rename(columns={0: 'Brottstyp'})

df_cleaned.head(10)

,Brottstyp,1,2,3,4,5
0,Anmälda brott,NaN,Brottsförebyggande rådet,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,År,År,År,År,År
3,NaN,2021,2022,2023,2024,2025
4,NaN,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN,NaN,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",1004,978,1017,1039,1072
9,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN


In [6]:
# Lägg till årtal i kolumn 1-5

years = [df_cleaned.iloc[3, i] for i in range(1, len(df_cleaned.columns))]

df_cleaned.columns = ['Brottstyp'] + years

df_cleaned.head(10)

,Brottstyp,2021,2022,2023,2024,2025
0,Anmälda brott,NaN,Brottsförebyggande rådet,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,År,År,År,År,År
3,NaN,2021,2022,2023,2024,2025
4,NaN,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN,NaN,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",1004,978,1017,1039,1072
9,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(df_cleaned.dtypes)

Brottstyp       str
2021         object
2022         object
2023         object
2024         object
2025         object
dtype: object


In [8]:
# Justera data type för brottsdatan per capita (str -> int) 

year_cols = ['2021', '2022', '2023', '2024', '2025']
df_cleaned[year_cols] = df_cleaned[year_cols].apply(pd.to_numeric, errors='coerce')

In [9]:
print(df_cleaned.dtypes)

Brottstyp        str
2021         float64
2022         float64
2023         float64
2024         float64
2025         float64
dtype: object


In [10]:
print(df_cleaned['Brottstyp'])

0                                         Anmälda brott
1                                                   NaN
2                                                   NaN
3                                                   NaN
4                                                   NaN
5                                      Stockholm kommun
6                             3-7 kap. Brott mot person
7                        3 kap. Brott mot liv och hälsa
8                        Misshandel inkl. grov (5, 6 §)
9                             3-7 kap. Brott mot person
10                     4 kap. Brott mot frihet och frid
11                            3-7 kap. Brott mot person
12                                   6 kap. Sexualbrott
13                      8-12 kap. Brott mot förmögenhet
14                               8 kap. Stöld, rån m.m.
15    Tillgrepp av motordrivet fortskaffningsmedel (...
16                      8-12 kap. Brott mot förmögenhet
17                               8 kap. Stöld, r

In [13]:
list(df_raw[0].dropna().unique())

['Anmälda brott',
 'Stockholm kommun',
 '3-7 kap. Brott mot person',
 '3 kap. Brott mot liv och hälsa',
 'Misshandel inkl. grov (5, 6 §)',
 '4 kap. Brott mot frihet och frid',
 '6 kap. Sexualbrott',
 '8-12 kap. Brott mot förmögenhet',
 '8 kap. Stöld, rån m.m.',
 'Tillgrepp av motordrivet fortskaffningsmedel (7 §)',
 'Stöld av icke motordrivet fortskaffningsmedel (1, 2, 4 §)',
 'Cykel',
 'Stöld genom inbrott, inbrottsstöld i bostad, ej av skjutvapen (1, 2, 4, 4 a§)',
 'Inbrottsstöld i bostad m.m.',
 'Övrig stöld (1, 2, 4 §)',
 'Från fordon m.m.',
 'Rån, inkl. grovt (5, 6 §)',
 '12 kap. Skadegörelsebrott',
 'Brott mot narkotikastrafflagen']

In [ ]:
df_cleaned = duckdb.sql("""--sql
SELECT
    CASE Brottstyp
        WHEN 'Misshandel inkl. grov (5, 6 §)' THEN 'Misshandel'
        WHEN '4 kap. Brott mot frihet och frid' THEN 'Hot & olaga intrång'
        WHEN '6 kap. Sexualbrott' THEN 'Sexualbrott'
        WHEN 'därav biltillgrepp' THEN 'Bilstöld'
        WHEN 'därav cykeltillgrepp' THEN 'Cykelstöld'
        WHEN 'därav stöld ur och från motordrivet fordon' THEN 'Stöld från fordon'
        WHEN 'därav inbrottsstöld i bostad' THEN 'Bostadsinbrott'
        WHEN 'därav rån, inkl. grovt' THEN 'Rån'
        WHEN '12 kap. Skadegörelsebrott' THEN 'Skadegörelse'
        WHEN 'Brott mot narkotikastrafflagen' THEN 'Narkotikabrott'
    END AS Brottstyp,
        * EXCLUDE (Brottstyp)              
FROM df_cleaned
WHERE Brottstyp IN (
    'därav misshandel inkl. grov',
    '4 kap. Brott mot frihet och frid',
    '6 kap. Sexualbrott',
    'därav biltillgrepp',
    'därav cykeltillgrepp',
    'därav stöld ur och från motordrivet fordon',
    'därav inbrottsstöld i bostad',
    'därav rån, inkl. grovt',
    '12 kap. Skadegörelsebrott',
    'Brott mot narkotikastrafflagen'
)
""").df()